## 🏗️ Notebook Execution Logic

Phases 1 & 2: Loading the Bangalore Graph
We import the road network. This defines the Spatial World—how 155k points are connected by roads. Without this, the model wouldn't know which neighborhoods are neighbors.

Phase 3: Model Initialization
We build the "Brain" (ST-PIGNN). It uses GINEConv to understand road shapes and a GRU to understand how pollution stays in the air over time.

Phase 4: Sensor Sanitization (The Filter)
This is a "Guardrail." We tell the model: "Out of 155k points, only trust these 23 points for the truth." This prevents the model from "cheating" and ensures it actually learns to predict, not just memorize.

Phase 5: The "Bunker" Audit
A final health check. We verify that features like Elevation (Topography) and Road Distances aren't broken or empty. If this fails, we don't waste GPU time training on bad data.

Phase 5.1: Scale Stabilization (The Fix)
This prevents Gradient Explosion. We scale PM2.5 and Distances down to decimals (0 to 1). This stops the math from getting too large and crashing the GPU with NaN errors.

Phase 6: The Forward-Pass Bridge
An "Adapter" layer. Since our model expects a sequence of data but our graph is currently a single "snapshot," this code translates the data so the model can process it.

Phase 7: Production Training (The Engine)
The heavy lifting. We run 100 epochs on the V100. We use Gradient Clipping here as a "Safety Shield" to keep the learning smooth and stable.

Phase 7.5: The Atomic Commit (The Safe)
We save the "Gold" weights and the Scale Factor (500) to a permanent file. This is your "Save Game" so you can close the notebook and still have your results.

In [1]:
# =================================================================
# PHASE 1: ENVIRONMENT SETUP & GPU VERIFICATION
# =================================================================
import torch
import os
import sys
import pandas as pd
import numpy as np
import types
import torch.nn.functional as F
from tqdm.auto import tqdm

# 1. Patch PyG dependencies to match HPC's Torch version & CUDA 12.x
# This prevents the 'GLIBC_2.32 not found' error during training.
v = torch.__version__.split('+')[0]
cu = "cu" + torch.version.cuda.replace('.', '')
print(f"🛠️ Patching PyG for Torch {v} on {cu}...")

# Run installation quietly to keep the notebook clean
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv \
    -f https://data.pyg.org/whl/torch-{v}+{cu}.html --quiet

# 2. Corrected Imports from your local physics_config
# Ensures we use the same constants for Bangalore's topography
try:
    sys.path.append(os.path.abspath('..'))
    from shared.physics_config import PHYSICS_LOSS_LAMBDA, RespiratoryCostant
    print("✅ Physics Config loaded successfully.")
except ImportError:
    print("⚠️ Physics Config not found at '..'. Using default constants.")
    PHYSICS_LOSS_LAMBDA = 0.1

# 3. Verify V100 Hardware & VRAM
# This is the "Bunker" check to ensure we aren't accidentally on a CPU node.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("-" * 50)
    print(f"🚀 HARDWARE: {torch.cuda.get_device_name(0)}")
    print(f"📊 TOTAL VRAM: {props.total_memory / 1024**3:.2f} GB")
    print(f"🔥 CUDA VERSION: {torch.version.cuda}")
    print("-" * 50)
else:
    print("❌ CRITICAL ERROR: CUDA NOT FOUND. Switch to a GPU Kernel immediately.")

# 4. Global Constants
# We use these to map the 155k nodes correctly in Phase 2
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("✅ Phase 1 Complete. System ready for Data Ingestion.")

🛠️ Patching PyG for Torch 2.8.0 on cu128...
✅ Physics Config loaded successfully.
--------------------------------------------------
🚀 HARDWARE: Tesla V100-PCIE-32GB
📊 TOTAL VRAM: 31.74 GB
🔥 CUDA VERSION: 12.8
--------------------------------------------------
✅ Phase 1 Complete. System ready for Data Ingestion.


In [2]:
# =================================================================
# PHASE 2: HIGH-FIDELITY DATA INGESTION & UNPACKING
# =================================================================
from torch_geometric.data import Data
import os

# 1. Absolute Path Verification
# We use the absolute path to ensure Jupyter doesn't get lost in subdirectories.
home = os.path.expanduser("~")
GRAPH_PATH = os.path.join(home, "Capstone/data/static_graph.pt")

print(f"📂 Loading Gold Standard Graph from: {GRAPH_PATH}")

# 2. Load the Object (Handling Dictionary vs Data Object)
raw_payload = torch.load(GRAPH_PATH, map_location='cpu', weights_only=False)

if isinstance(raw_payload, dict):
    print("📦 Dictionary detected. Unpacking to PyG Data object...")
    graph = Data(
        x=raw_payload['x'], 
        edge_index=raw_payload['edge_index'], 
        edge_attr=raw_payload['edge_attr']
    )
    # Persist the schema metadata for reference
    graph.node_schema = raw_payload.get('node_feature_schema', [])
    graph.edge_schema = raw_payload.get('edge_attr_schema', [])
else:
    print("💎 Raw PyG Data object detected.")
    graph = raw_payload

# 3. Data Surgery: Elevation Restoration (The "12m Std Dev" Fix)
# Ensures Bangalore isn't treated as a flat plane.
elev_idx = 7
if graph.x[:, elev_idx].std() < 1.0:
    print("🏗️ Restoring Elevation Topography...")
    min_e, max_e = graph.x[:, elev_idx].min(), graph.x[:, elev_idx].max()
    # Stretch to realistic Bangalore range [890m, 960m]
    graph.x[:, elev_idx] = 890 + (graph.x[:, elev_idx] - min_e) * (70 / (max_e - min_e + 1e-6))

# 4. Final Cleanup: NaN Removal
# Safety check to ensure no stray NaNs survived the push.
if torch.isnan(graph.x).any():
    print("⚠️ Stray NaNs detected. Applying zero-fill.")
    graph.x = torch.nan_to_num(graph.x, nan=0.0)

print("-" * 50)
print(f"✅ DATA INGESTED SUCCESSFULLY")
print(f"📊 Nodes: {graph.num_nodes:,}")
print(f"🛣️ Edges: {graph.num_edges:,}")
print(f"🧬 Node Features: {graph.x.shape[1]}")
print(f"🚧 Edge Features: {graph.edge_attr.shape[1]}")
print("-" * 50)

/home/jupyter-1nt23cb058/.local/lib/python3.9/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /lib/x86_64-linux-gnu/libc.so.6: version `GLIBC_2.32' not found (required by /home/jupyter-1nt23cb058/.local/lib/python3.9/site-packages/torch_scatter/_version_cuda.so)
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/home/jupyter-1nt23cb058/.local/lib/python3.9/site-packages/torch_geometric/typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: /lib/x86_64-linux-gnu/libc.so.6: version `GLIBC_2.32' not found (required by /home/jupyter-1nt23cb058/.local/lib/python3.9/site-packages/torch_cluster/_version_cuda.so)
  warnings.warn(f"An issue occurred while importing 'torch-cluster'. "
/home/jupyter-1nt23cb058/.local/lib/python3.9/site-packages/torch_geometric/typing.py:113: UserWarning: An issue occurred while importing 'torch-s

📂 Loading Gold Standard Graph from: /home/jupyter-1nt23cb058/Capstone/data/static_graph.pt
📦 Dictionary detected. Unpacking to PyG Data object...
🏗️ Restoring Elevation Topography...
--------------------------------------------------
✅ DATA INGESTED SUCCESSFULLY
📊 Nodes: 154,902
🛣️ Edges: 424,550
🧬 Node Features: 17
🚧 Edge Features: 16
--------------------------------------------------


In [3]:
# =================================================================
# PHASE 3 pre: COMPREHENSIVE "BUNKER" INTEGRITY AUDIT
# =================================================================
import numpy as np

print("--- 🛡️ FINAL DATA TELEMETRY CHECK ---")

# 1. Dead-Zone Ghost Check (Crucial for GNN stability)
zero_nodes = (graph.x.abs().sum(dim=1) == 0).sum().item()
zero_ratio = zero_nodes / graph.num_nodes
print(f"🚫 Dead-Zone Ratio: {zero_ratio:.8f} (Target: 0.00000000)")

# 2. Micro-Scale Topography (Index 7: Elevation)
elev_std = graph.x[:, 7].std().item()
elev_range = graph.x[:, 7].max().item() - graph.x[:, 7].min().item()
print(f"⛰️ Elevation Std Dev: {elev_std:.2f}m | Range: {elev_range:.2f}m")

# 3. Spatial Signal Check (Correlation)
# Does the PM2.5 (Index 0) relate to the Elevation (Index 7)?
pm25_np = graph.x[:, 0].cpu().numpy()
elev_np = graph.x[:, 7].cpu().numpy()
correlation = np.corrcoef(pm25_np, elev_np)[0, 1]
print(f"📊 PM2.5/Elevation Correlation: {correlation:.4f}")

# 4. Edge Complexity (Index 0 of Edge_Attr: Distance)
avg_dist = graph.edge_attr[:, 0].mean().item()
max_dist = graph.edge_attr[:, 0].max().item()
print(f"🛣️ Avg Road Segment: {avg_dist:.2f}m | Max: {max_dist:.2f}m")

# 5. Infrastructure Diversity (Indices 1-15: One-Hot Road Types)
# Ensure we didn't lose the road categories during unpacking
road_type_variance = graph.edge_attr[:, 1:].std().item()
print(f"🚧 Road-Type Variance: {road_type_variance:.4f} (Target: > 0)")

print("-" * 50)
if zero_ratio == 0 and elev_std > 10 and abs(correlation) > 0.01:
    print("🚀 AUDIT PASSED: DATA IS HIGH-FIDELITY. PROCEED TO MASKING.")
else:
    print("❌ AUDIT FAILED: Data remains unphysical. Check Phase 2 surgery.")

--- 🛡️ FINAL DATA TELEMETRY CHECK ---
🚫 Dead-Zone Ratio: 0.00000000 (Target: 0.00000000)
⛰️ Elevation Std Dev: 12.06m | Range: 70.00m
📊 PM2.5/Elevation Correlation: 0.2307
🛣️ Avg Road Segment: 65.67m | Max: 6884.93m
🚧 Road-Type Variance: 0.2494 (Target: > 0)
--------------------------------------------------
🚀 AUDIT PASSED: DATA IS HIGH-FIDELITY. PROCEED TO MASKING.


In [4]:
# =================================================================
# PHASE 3: ST-PIGNN INITIALIZATION & TRAINING HARNESS
# =================================================================
import torch
from gnn.model import STPIGNN

# 1. Architecture Configuration (From 155k-Node Audit)
# Node Features: 17 | Edge Attributes: 16
NODE_IN = graph.x.shape[1]
EDGE_IN = graph.edge_attr.shape[1]

SPATIAL_HIDDEN = 256
TEMPORAL_HIDDEN = 256
GNN_LAYERS = 3       # Depth for complex Bangalore intersections
GRU_LAYERS = 2       # Temporal persistence for the sequence
DROPOUT = 0.1

# 2. Model Initialization
model = STPIGNN(
    node_in_dim=NODE_IN,
    edge_dim=EDGE_IN,
    spatial_hidden_dim=SPATIAL_HIDDEN,
    temporal_hidden_dim=TEMPORAL_HIDDEN,
    gnn_layers=GNN_LAYERS,
    gru_layers=GRU_LAYERS,
    dropout=DROPOUT
).to(device)

# 3. Stabilized Training Harness
# We use 1e-4 for the learning rate to ensure smooth convergence on 155k nodes.
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

# Using the non-deprecated Scaler for Torch 2.x
scaler = torch.amp.GradScaler('cuda')

print("-" * 50)
print(f"🚀 ST-PIGNN Initialized on {device}")
print(f"🧬 Node Dim: {NODE_IN} | 🚧 Edge Dim: {EDGE_IN}")
print(f"📦 Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"⚖️ Scaler: torch.amp.GradScaler('cuda') [Ready]")
print("-" * 50)

--------------------------------------------------
🚀 ST-PIGNN Initialized on cuda
🧬 Node Dim: 17 | 🚧 Edge Dim: 16
📦 Parameters: 1,203,713
⚖️ Scaler: torch.amp.GradScaler('cuda') [Ready]
--------------------------------------------------


In [5]:
# =================================================================
# PHASE 4: SENSOR SANITIZATION & DEVICE SYNC
# =================================================================
import torch
import torch.nn.functional as F

# 1. Ensure the Graph and its features are on the V100 FIRST
graph = graph.to(device)

# 2. Identify where PM2.5 data exists in the feature matrix (Index 0)
pm25_vals = graph.x[:, 0]
potential_nodes = torch.where((pm25_vals > 0.1) & (pm25_vals < 500.0))[0]

# 3. Strict Selection: Lock in exactly 23 coordinates
# Using a fixed seed (42) to ensure reproducibility across runs
torch.manual_seed(42) 
perm = torch.randperm(len(potential_nodes))
sensor_indices = potential_nodes[perm[:23]]

# 4. Create the Precision Mask on the SAME device as the graph (V100)
train_mask = torch.zeros(graph.num_nodes, dtype=torch.bool, device=device)
train_mask[sensor_indices] = True

# 5. Verification Output
sensor_mean = graph.x[train_mask, 0].mean().item()
print(f"✅ SANITIZATION COMPLETE")
print(f"📍 Total Nodes: {graph.num_nodes:,}")
print(f"🎯 Training Anchors (Physical Sensors): {train_mask.sum().item()}")
print(f"🔍 Sensor PM2.5 Mean: {sensor_mean:.2f}")
print(f"🖥️ Device Alignment: {graph.x.device}")

✅ SANITIZATION COMPLETE
📍 Total Nodes: 154,902
🎯 Training Anchors (Physical Sensors): 23
🔍 Sensor PM2.5 Mean: 59.33
🖥️ Device Alignment: cuda:0


In [6]:
# =================================================================
# PHASE 5: THE "BUNKER" FEATURE AUDIT
# =================================================================
import pandas as pd
import numpy as np

def perform_deep_audit(tensor, name):
    print(f"\n--- {name} Audit ---")
    # Move to CPU just for the audit stats calculation
    data = tensor.detach().cpu().numpy()
    stats = []
    for i in range(data.shape[1]):
        col = data[:, i]
        stats.append({
            "Dim": i,
            "Min": np.nanmin(col),
            "Max": np.nanmax(col),
            "Std": np.nanstd(col),
            "Zeros (%)": (col == 0).sum() / len(col) * 100
        })
    return pd.DataFrame(stats)

# Audit Node Features (17) and Edge Attributes (16)
node_stats = perform_deep_audit(graph.x, "Node Features")
print(node_stats.to_string(index=False))

edge_stats = perform_deep_audit(graph.edge_attr, "Edge Attributes")
print("\n" + edge_stats.to_string(index=False))

# 🏁 FINAL VERIFICATION LOGIC
topography_ok = graph.x[:, 7].std() > 10.0
pm25_ok = graph.x[:, 0].max() < 1000.0
anchors_ok = train_mask.sum().item() == 23

print("\n" + "="*30)
if topography_ok and pm25_ok and anchors_ok:
    print("🚀 SYSTEM IS GREEN. DATA IS SANITIZED. PROCEED TO PHASE 6.")
else:
    print("❌ VERIFICATION FAILED. DO NOT PROCEED.")
    if not topography_ok: print(f"Reason: Elevation Std is too low.")
    if not anchors_ok: print(f"Reason: Anchor count is {train_mask.sum().item()}, not 23.")


--- Node Features Audit ---
 Dim        Min         Max        Std  Zeros (%)
   0  15.480000  171.309998  29.406353        0.0
   1  16.061230  342.935608  53.778126        0.0
   2  14.010000   75.900002   8.806015        0.0
   3   2.540000   12.700000   1.881796        0.0
   4   0.300000 1886.721558 479.156250        0.0
   5   1.200000    4.320000   0.586757        0.0
   6   2.000000  114.000000  33.168144        0.0
   7 890.000000  960.000000  12.059396        0.0
   8  18.200001   28.900000   1.676639        0.0
   9  23.000000   95.000000  13.273940        0.0
  10 910.299988  916.299988   0.990733        0.0
  11   7.000000   44.200001   8.026325        0.0
  12   3.300000   15.600000   2.538717        0.0
  13   7.200000   30.900000   6.418898        0.0
  14  22.200001   33.599998   3.430664        0.0
  15 218.000000  988.000000 172.486877        0.0
  16 800.000000 1000.000000  41.454891        0.0

--- Edge Attributes Audit ---

 Dim      Min         Max       Std  Ze

In [7]:
# =================================================================
# PHASE 5.1: SURGICAL NORMALIZATION (EXPLOSION PREVENTION)
# =================================================================
import torch

print("🛡️ Applying Scale Stabilization...")

# 1. Normalize Node Features (Dim 0: PM2.5 and Dim 7: Elevation)
# We use simple Max-Scaling to keep them in [0, 1] range
with torch.no_grad():
    # Normalize PM2.5 (Targets)
    graph.x[:, 0] = graph.x[:, 0] / 500.0 
    
    # Normalize Elevation (Topography)
    elev_min = graph.x[:, 7].min()
    elev_max = graph.x[:, 7].max()
    graph.x[:, 7] = (graph.x[:, 7] - elev_min) / (elev_max - elev_min + 1e-6)
    
    # 2. Normalize Edge Attributes (Dim 0: Distance)
    # Distances go up to 6.8km; we must scale this down!
    dist_max = graph.edge_attr[:, 0].max()
    graph.edge_attr[:, 0] = graph.edge_attr[:, 0] / dist_max

print(f"✅ Normalization Complete.")
print(f"📊 New Node Max: {graph.x.max().item():.4f} | New Edge Max: {graph.edge_attr.max().item():.4f}")

🛡️ Applying Scale Stabilization...
✅ Normalization Complete.
📊 New Node Max: 1886.7216 | New Edge Max: 1.0000


In [8]:
# SANITY CHECK
print(f"Target PM2.5 Range: {graph.x[train_mask, 0].min().item():.4f} to {graph.x[train_mask, 0].max().item():.4f}")

Target PM2.5 Range: 0.0310 to 0.1760


In [9]:
# =================================================================
# PHASE 6: FORWARD-PASS BRIDGE (DYNAMIC LAYER DISCOVERY)
# =================================================================
import types
import torch.nn as nn
import torch.nn.functional as F

# 1. DYNAMIC DISCOVERY: Find the Regressor and Input Projection
proj_layer = None
final_layer = None

for name, module in model.named_modules():
    # Find the 17 -> 256 projection
    if isinstance(module, nn.Linear) and module.in_features == 17 and module.out_features == 256:
        proj_layer = module
    # Find the final PM2.5 regressor (Output dimension is 1)
    if isinstance(module, nn.Linear) and module.out_features == 1:
        final_layer = module

if proj_layer is None or final_layer is None:
    print("⚠️ Warning: Auto-discovery failed. Check linear layer dimensions.")

def patched_forward(self, x, edge_index, edge_attr):
    # Shape Normalization
    if x.dim() == 2:
        x = x.unsqueeze(0).unsqueeze(0)
    
    bsz, seq_len, num_nodes, _ = x.shape
    x_t = x[:, 0, :, :].view(-1, x.size(-1))
    
    # Use discovered projection
    x_t = proj_layer(x_t)
    
    # Spatial Pass
    for conv, norm in zip(self.gnn_layers, self.norms):
        residual = x_t
        x_t = conv(x_t, edge_index, edge_attr)
        x_t = norm(x_t)
        x_t = F.relu(x_t)
        if x_t.shape == residual.shape:
            x_t = x_t + residual
            
    # Temporal Pass
    x_t = x_t.unsqueeze(1) 
    gru_out, _ = self.gru(x_t)
    
    # Use discovered final layer
    out = final_layer(gru_out[:, -1, :])
    return out.view(bsz, num_nodes, -1)

# Apply the dynamic patch
model.forward = types.MethodType(patched_forward, model)
print(f"✅ Discovery Complete: Proj={proj_layer} | Final={final_layer}")
print("✅ Forward-Pass Bridge updated with dynamic attributes.")

✅ Discovery Complete: Proj=Linear(in_features=17, out_features=256, bias=True) | Final=Linear(in_features=256, out_features=1, bias=True)
✅ Forward-Pass Bridge updated with dynamic attributes.


In [10]:
# =================================================================
# FINAL SYSTEM VERIFICATION (STRICT)
# =================================================================
import torch

print("--- 🛰️ FINAL SYSTEM TELEMETRY ---")

# 1. Shape Consistency Check
try:
    with torch.no_grad():
        test_out = model(graph.x, graph.edge_index, graph.edge_attr)
    
    print(f"✅ Forward Pass: SUCCESS")
    print(f"📊 Output Shape: {test_out.shape} (Target: [1, 154902, 1])")
    
    # 2. Supervision Alignment
    # Ensure the 23 anchors align with the output tensor
    masked_out = test_out.squeeze()[train_mask]
    masked_target = graph.x[train_mask, 0]
    print(f"🎯 Supervision: {masked_out.shape[0]} predicted points vs {masked_target.shape[0]} targets.")
    
    # 3. Gradient Flow Verification
    model.train()
    optimizer.zero_grad(set_to_none=True)
    with torch.amp.autocast('cuda'):
        verify_out = model(graph.x, graph.edge_index, graph.edge_attr)
        verify_loss = F.mse_loss(verify_out.squeeze()[train_mask], graph.x[train_mask, 0])
    
    scaler.scale(verify_loss).backward()
    
    # Check if gradients actually exist in the first GNN layer
    first_layer_grad = next(model.gnn_layers[0].parameters()).grad
    grad_norm = first_layer_grad.norm().item() if first_layer_grad is not None else 0
    
    print(f"📉 Initial Loss: {verify_loss.item():.6f}")
    print(f"⚡ Gradient Flow: {'DETECTED' if grad_norm > 0 else 'BLOCKED'}")
    print(f"📏 Gradient Norm: {grad_norm:.8f}")

    # 4. Device Integrity
    print(f"🖥️  GPU VRAM: {torch.cuda.memory_allocated() / 1024**2:.2f} MB / 32GB")

except Exception as e:
    print(f"❌ SYSTEM FAILURE: {e}")
    raise e

print("\n" + "="*40)
if grad_norm > 0 and not torch.isnan(verify_loss):
    print("🚀 ALL SYSTEMS NOMINAL. READY FOR PRODUCTION TRAINING.")
else:
    print("❌ CRITICAL ERROR: Gradient death or NaN loss. DO NOT TRAIN.")

--- 🛰️ FINAL SYSTEM TELEMETRY ---
✅ Forward Pass: SUCCESS
📊 Output Shape: torch.Size([1, 154902, 1]) (Target: [1, 154902, 1])
🎯 Supervision: 23 predicted points vs 23 targets.
📉 Initial Loss: 0.016940
⚡ Gradient Flow: DETECTED
📏 Gradient Norm: 641.40209961
🖥️  GPU VRAM: 74.40 MB / 32GB

🚀 ALL SYSTEMS NOMINAL. READY FOR PRODUCTION TRAINING.


In [15]:
# =================================================================
# PHASE 7: PRODUCTION ENGINE (RESUMABLE & STABILIZED)
# =================================================================
import os
import torch
from tqdm.auto import tqdm

# 1. Config & Paths
EPOCHS = 100
CHECKPOINT_PATH = '../cache/stpignn_bangalore_v1.pt'
GOLD_PATH = '../models/stpignn_bangalore_gold_v1.pt'
os.makedirs('../cache', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# 2. Resumability Logic
start_epoch = 1
history = []
if os.path.exists(CHECKPOINT_PATH):
    resume = input(f"🛰️ Checkpoint found. Resume from last saved state? (y/n): ").lower()
    if resume == 'y':
        print("🔄 Loading checkpoint...")
        ckpt = torch.load(CHECKPOINT_PATH)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        start_epoch = ckpt['epoch'] + 1
        history = ckpt.get('history', [])
        print(f"✅ Resuming from Epoch {start_epoch}")
    else:
        print("🆕 Starting fresh training...")
else:
    print("🚀 No checkpoint found. Starting fresh...")

# 3. Training Loop
model.train()
pbar = tqdm(range(start_epoch, EPOCHS + 1), desc="🛰️ Bangalore GNN Training", unit="epoch")

for epoch in pbar:
    optimizer.zero_grad(set_to_none=True)
    
    with torch.amp.autocast('cuda'):
        predictions = model(graph.x, graph.edge_index, graph.edge_attr)
        loss = F.mse_loss(predictions.squeeze()[train_mask], graph.x[train_mask, 0])
    
    scaler.scale(loss).backward()
    
    # 🛡️ Gradient Shield
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    scaler.step(optimizer)
    scaler.update()
    
    current_loss = loss.item()
    history.append(current_loss)
    pbar.set_postfix({"MSE": f"{current_loss:.6f}", "VRAM": f"{torch.cuda.memory_allocated()/1024**2:.1f}MB"})
    
    # 💾 Auto-Checkpoint (Every 10 Epochs)
    if epoch % 10 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': current_loss,
            'history': history
        }, CHECKPOINT_PATH)

# 4. Final Gold Commit (The old 7.5 logic)
commit_payload = {
    'epoch_final': EPOCHS,
    'model_state_dict': model.state_dict(),
    'scaling_factor': 500.0,
    'final_mse': history[-1],
    'train_mask': train_mask.cpu(),
    'node_features_dim': 17,
    'edge_features_dim': 16
}
torch.save(commit_payload, GOLD_PATH)

print("-" * 30)
print(f"✅ MISSION SUCCESS: Gold Model secured at {GOLD_PATH}")

🛰️ Checkpoint found. Resume from last saved state? (y/n): yes
🆕 Starting fresh training...


🛰️ Bangalore GNN Training:   0%|          | 0/100 [00:00<?, ?epoch/s]

------------------------------
✅ MISSION SUCCESS: Gold Model secured at ../models/stpignn_bangalore_gold_v1.pt
